In [ ]:
import requests
import pandas as pd
from datetime import datetime

## Identifiants STRAVA

In [ ]:
CLIENT_ID =
CLIENT_SECRET =
REFRESH_TOKEN =

In [ ]:
# curl -X POST https://www.strava.com/api/v3/oauth/token \n#   -d client_id=YOUR_CLIENT_ID \n#   -d client_secret=YOUR_CLIENT_SECRET \n#   -d code=YOUR_AUTH_CODE \n#   -d grant_type=authorization_code


In [ ]:
# {"token_type":"Bearer","expires_at":...,"refresh_token":"YOUR_REFRESH_TOKEN","access_token":"YOUR_ACCESS_TOKEN",...}


In [ ]:
# http://localhost/exchange_token?state=&code=YOUR_AUTH_CODE&scope=read,activity:read_all


## Access Token

In [ ]:
def get_access_token():
    url = "https://www.strava.com/oauth/token"

    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "refresh_token": REFRESH_TOKEN,
        "grant_type": "refresh_token"
    }

    response = requests.post(url, data=payload)
    response.raise_for_status()

    tokens = response.json()
    return tokens["access_token"]

In [ ]:
access_token = get_access_token()
print(access_token[:20])

## Liste d'activités

In [ ]:
def get_activities(access_token, per_page=50, page=1):
    url = "https://www.strava.com/api/v3/athlete/activities"

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    params = {
        "per_page": per_page,
        "page": page
    }

    response = requests.get(url, headers=headers, params=params)

    print(response.status_code)
    print(response.text)
    response.raise_for_status()

    return response.json()

In [ ]:
activities = get_activities(access_token)
len(activities)

In [ ]:
df = pd.json_normalize(activities)
df.head()

In [ ]:
print(df.columns)

In [ ]:
# cols = [
#     "id",
#     "name",
#     "type",
#     "distance",
#     "moving_time",
#     "total_elevation_gain",
#     "start_date",
#     "average_speed",
#     "average_heartrate",
#     "map.summary_polyline"
# ]

# df[cols].head()

In [ ]:
def get_activity_streams(activity_id, access_token):
    url = f"https://www.strava.com/api/v3/activities/{activity_id}/streams"

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    params = {
        "keys": "time,latlng,distance,altitude,heartrate,velocity_smooth",
        "key_by_type": "true"
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    return response.json()

In [ ]:
tracks = []

for activity_id in df["id"]:
    streams = get_activity_streams(activity_id, access_token)

    # récupérer les coordonnées GPS
    coords = streams["latlng"]["data"]
    print(activity_id, coords[:5])  # aperçu des 5 premiers points

    track_df = pd.DataFrame(coords, columns=["lat", "lng"])
    track_df["activity_id"] = activity_id

    tracks.append(track_df)

tracks_df = pd.concat(tracks, ignore_index=True)

In [ ]:
track_df = pd.DataFrame(coords, columns=["lat", "lng"])
track_df["altitude"] = streams["altitude"]["data"]
track_df["distance"] = streams["distance"]["data"]
track_df["time"] = streams["time"]["data"]

track_df

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot( track_df['distance'],track_df['altitude'])